# GPT-2 Model Training!

Train a distilGPT-2 model on the corbt/all-recipes dataset, to make your own special (stupid) cookgpt. I intend this to run on a Colab runtime, or you can run it locally if you want the pain. (I trained it ok enough on a M1 Pro, but it was slower than the google TPU v5e1, so I recommend Colab.)

In [ ]:
# StupidCookTrainer Notebook
# I suggest running this notebook on a Colab runtime since
# I assume that you don't have a H100 or a v5e1 TPU.
# colab.research.google.com --- head here
# - 0681691

# 1, Imports (refer to requirements.txt to install)
# Note; this will take up a lot of your storage. So run it on colab.

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

# dataset imports
from datasets import (load_dataset, Dataset)
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 6767 # seed for everything following (67 lol)

print(f"Using device: {DEVICE}") # - check ur device

In [ ]:
# 2. load the dataset

TRAIN_SIZE = 100000 # @param {type:"integer"}
EVAL_SIZE = 5000 # @param {type:"integer"}

raw_dataset = load_dataset("corbt/all-recipes")

# Handle either DatasetDict with train split or single split datasets.
if "train" in raw_dataset:
    base_dataset = raw_dataset["train"]
else:
    first_key = list(raw_dataset.keys())[0]
    base_dataset = raw_dataset[first_key]

dataset = base_dataset.train_test_split(test_size=0.1, seed=SEED)

train_size = min(TRAIN_SIZE, len(dataset["train"]))
eval_size = min(EVAL_SIZE, len(dataset["test"]))

train_split = dataset["train"].shuffle(seed=SEED).select(range(train_size))
eval_split = dataset["test"].shuffle(seed=SEED).select(range(eval_size))

print(f"Train samples: {len(train_split):,}")
print(f"Eval samples: {len(eval_split):,}")

SyntaxError: invalid syntax (895487332.py, line 3)

In [ ]:
# 3. load distillGPT2 and tokenize everything

MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

def tokenize(batch):
    return tokenizer(batch["input"], truncation=True)

train_tokens = train_split.map(tokenize, batched=True, remove_columns=train_split.column_names)
eval_tokens = eval_split.map(tokenize, batched=True, remove_columns=eval_split.column_names)

print(f"Tokenized train rows: {len(train_tokens):,}")
print(f"Tokenized eval rows: {len(eval_tokens):,}")

In [ ]:
# 4. pack the sequences into 512 token blocks

BLOCK_SIZE = 512

def pack_sequences(tokenized_dataset):
    all_ids = []

    for sample in tokenized_dataset:
        all_ids.extend(sample["input_ids"] + [tokenizer.eos_token_id])

    # Drop the tail if it's shorter than BLOCK_SIZE to keep tensor shapes uniform.
    total_length = (len(all_ids) // BLOCK_SIZE) * BLOCK_SIZE
    all_ids = all_ids[:total_length]

    chunks = [
        all_ids[i:i + BLOCK_SIZE]
        for i in range(0, total_length, BLOCK_SIZE)
    ]

    return {
        "input_ids": chunks,
        "labels": [c[:] for c in chunks],
    }

train_packed = pack_sequences(train_tokens)
eval_packed = pack_sequences(eval_tokens)

train_dataset = Dataset.from_dict(train_packed)
eval_dataset = Dataset.from_dict(eval_packed)

print(f"Packed train blocks: {len(train_dataset):,}")
print(f"Packed eval blocks: {len(eval_dataset):,}")

In [ ]:
# 5. data collator, training args, trainer
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="./rcpmodel",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    warmup_steps=200,
    save_steps=500,
    save_total_limit=2,
    save_only_model=True,
    save_safetensors=True,
    logging_steps=100,
    eval_steps=500,
    fp16=use_fp16,
    report_to="none",
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

In [ ]:
# 6. train and save weights
train_result = trainer.train()

save_dir = "./rcpmodel/final-weights"
model.save_pretrained(save_dir, safe_serialization=True)

print("Training finished.")
print(f"Model weights saved to: {save_dir}")
print(train_result)

In [ ]:
# 7. generate a sample recipe
prompt = "Recipe for a cozy garlic mushroom pasta:\n\nIngredients:\n"

gen_inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
model.eval()

with torch.no_grad():
    generated = model.generate(
        **gen_inputs,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

sample_recipe = tokenizer.decode(generated[0], skip_special_tokens=True)
print(sample_recipe)